# Run notebook with prompts

1. Click **Run all** on this notebook.
2. Fill in the prompts and click **Submit & Run**.
3. The main notebook runs from top to bottom with your values.

**Setup (once):** in the main notebook, mark the first cell as a *Parameters cell* (cell `...` menu → *Toggle parameter cell*) containing:
```python
THISRUN_REPORTING_DATE = "2024-01-31"
THISRUN_ENVIRONMENT    = "dev"
THISRUN_CONFIG_SET     = "default"
THISRUN_CONTEXT_ID     = "PROD"
```
Then set `MAIN_NOTEBOOK` below to that notebook's name.

In [ ]:
import ipywidgets as widgets
from IPython.display import display
from datetime import date
from notebookutils import mssparkutils

# ---- Settings ----------------------------------------------------
MAIN_NOTEBOOK   = "main_notebook"   # name or path, e.g. "/Migration/main_notebook"
TIMEOUT_SECONDS = 3600              # max run time for the main notebook

# ---- Prompts -----------------------------------------------------
style = {"description_width": "160px"}
layout = widgets.Layout(width="420px")

reporting_date = widgets.DatePicker(description="Reporting date:", value=date.today(),
                                    style=style, layout=layout)
environment    = widgets.Dropdown(description="Environment:", options=["dev", "test", "prod"],
                                  value="dev", style=style, layout=layout)
config_set     = widgets.Text(description="Configuration set:", value="default",
                              style=style, layout=layout)
context_id     = widgets.Text(description="Context ID:", value="PROD",
                              style=style, layout=layout)

submit = widgets.Button(description="Submit & Run", button_style="success", icon="play")
output = widgets.Output()

In [ ]:
def on_submit(_):
    output.clear_output()
    with output:
        # Basic validation
        if reporting_date.value is None:
            print("Please choose a reporting date.")
            return
        if not config_set.value.strip() or not context_id.value.strip():
            print("Configuration set and Context ID cannot be empty.")
            return

        params = {
            "THISRUN_REPORTING_DATE": reporting_date.value.isoformat(),
            "THISRUN_ENVIRONMENT":    environment.value,
            "THISRUN_CONFIG_SET":     config_set.value.strip(),
            "THISRUN_CONTEXT_ID":     context_id.value.strip(),
        }

        print(f"Running '{MAIN_NOTEBOOK}' with:")
        for k, v in params.items():
            print(f"  {k} = {v}")

        submit.disabled = True          # prevent double runs
        submit.description = "Running..."
        try:
            result = mssparkutils.notebook.run(MAIN_NOTEBOOK, TIMEOUT_SECONDS, params)
            print("\nFinished successfully.")
            if result:
                print(f"Exit value: {result}")
        except Exception as e:
            # Errors inside button callbacks are otherwise silent, so show them here
            print(f"\nRun failed: {e}")
        finally:
            submit.disabled = False
            submit.description = "Submit & Run"

submit.on_click(on_submit)
display(widgets.VBox([reporting_date, environment, config_set, context_id, submit, output]))